<a href="https://colab.research.google.com/github/anandita-3217/EAI-Assignments/blob/main/EAI-Assignments%20/Assignment-2/SE26MAID034_EAI_Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Q1. When actions have different costs in a state based search space, an obvious choice is to use best-first search where the evaluation function is the cost of the path from the root to the current node. This is called Dijkstra’s algorithm by the theoretical computer science community, and uniform-cost search Uniform-cost search by the AI community. Implement the Dijkstra’s algorithm to all the cities in India and their Road distances. This info may be taken from open sources.

In [1]:
import heapq
import numpy as np
import time

In [2]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d kbdharun/a-star-algorithm-route-planning-dataset

!unzip -o a-star-algorithm-route-planning-dataset.zip -d indian-cities-dataset

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/kbdharun/a-star-algorithm-route-planning-dataset
License(s): MIT
100% 804/804 [00:00<00:00, 1.80MB/s]

Archive:  a-star-algorithm-route-planning-dataset.zip
  inflating: indian-cities-dataset/indian-cities-dataset.csv  


In [5]:
import pandas as pd

distances = pd.read_csv('a-star-algorithm-route-planning-dataset/indian-cities-dataset.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'a-star-algorithm-route-planning-dataset/indian-cities-dataset.csv'

In [ ]:
def dijkstra(graph, src):
  """
  dijkstra aka uniform cost search
  """
  n = len(graph)
  dist = [float('inf')] * n
  dist[src] = 0
  prev = [None] * n
  pq = [(0,src)]
  while pq:
    d, u = heapq.heappop(pq)
    if d > dist[u]:
      continue
    for v in range(n):
      w = graph[u][v]
      if u != v and w > 0:

        if dist[u] + w < dist[v]:
          dist[v] = dist[u]+w
          prev[v] = u
          heapq.heappush(pq, (dist[v],v))
  return dist, prev

In [ ]:
def dist_path(prev, target):
  path = []
  while target is not None:
    path.append(target)
    target = prev[target]
  return path[::-1]

In [ ]:
cities = ["Delhi", "Mumbai", "Bengaluru", "Hyderabad", "Chennai", "Kolkata"]
city_index = {name: i for i, name in enumerate(cities)}

Cities_Distances = [
    [0, 1419, 2121, 1499, 2107, 1453],      # Delhi
    [1419, 0, 984, 711, 1290, 1939],        # Mumbai
    [2121, 984, 0, 575, 346, 1875],         # Bengaluru
    [1499, 711, 575, 0, 633, 1492],         # Hyderabad
    [2107, 1290, 346, 633, 0, 1684],        # Chennai
    [1453, 1939, 1875, 1492, 1684, 0]       # Kolkata
]

src_city = "Delhi"
distances, prev = dijkstra(Cities_Distances, city_index[src_city])


In [ ]:
for city in cities:
    idx = city_index[city]
    path = [cities[i] for i in dist_path(prev, idx)]
    print(f"{city}: {distances[idx]} km  (path: {' -> '.join(path)})")

Delhi: 0 km  (path: Delhi)
Mumbai: 1419 km  (path: Delhi -> Mumbai)
Bengaluru: 2074 km  (path: Delhi -> Hyderabad -> Bengaluru)
Hyderabad: 1499 km  (path: Delhi -> Hyderabad)
Chennai: 2107 km  (path: Delhi -> Chennai)
Kolkata: 1453 km  (path: Delhi -> Kolkata)


Q2. An Unmanned Ground Vehicle (UGV) is a robot that finds the optimal path from a given user-specified start node to a user-specified Goal node on a map of a small area in a battlefield (Eg 70x70 Kms). There are obstacles known a-priori. The density of the obstacles can be generated randomly with three different levels of density. Design an algorithm that makes the UGV to navigate through this grid space avoiding all the known obstacles to reach the goal by the shortest distance. Trace this path along with the Measures of Effectiveness.

> Battlefield 70 * 70 with 1 representing obstacles and 0 free cells   

In [ ]:
def battle_field(size = 70, density_lvl= 'med', seed = None, start=None, goal=None):
  """
  Generates a battle field givee the size with randomly placed objects.
  Parameters:
    size (int): grid is size x size (default 70x70 km)
    density_lvl (str): 'low', 'med', or 'high' obstacle density
    start (tuple: Int): row, col of start cell — guaranteed free if given
    goal (tuple: Int): row, col of goal cell — guaranteed free if given
    seed (int): optional seed for reproducible grids

  Returns:
    2D numpy array, 0 = free, 1 = obstacle
  """
  if seed is not None:
    np.random.seed(seed)

  density = {'low': 0.10, 'med': 0.25, 'high': 0.40}
  probability = density.get(density_lvl.lower(),0.25)

  grid = (np.random.rand(size,size) < probability).astype(int)

  if start is not None:
    grid[start] = 0
  if goal is not None:
    grid[goal] = 0

  return grid


> The problem can be solved by using the A*

In [8]:
def _manhattan(a,b):
  """
  utility function that returns the manhattan distance given 2 points
  Manhattan distance gives the  length of the shortest path under
  grid constraints

  Parameter:
  a (tuple: Int): row column pair of point a
  b (tuple: Int): row column pair of point b

  Returns:
    Manhattan distance gives the  length of the shortest path under
  grid constraints
  """
  return abs(a[0]-b[0]) + abs(a[1]-b[1])

In [ ]:
def astar(grid, start, goal):
  """
  Given a grid, start, goal this function finds the solution if it exists based on the a* algorithm

  Parameters:
    grid (2D array): grid[row][col] == 0 is free, == 1 is obstacle
    start (tuple: Int): row, column pair of the start cell
    goal (tuple: Int): row, column pair of the goal cell

    Returns:
      list of (row, col) tuples from start to goal, or [] if no path exists

  """
  n_rows, n_cols = len(grid), len(grid[0])
  pq = []
  g_scores = {}
  came_from = {}
  visited = set()

  g_scores[start] = 0
  heapq.heappush(pq, (_manhattan(start,goal), start))
  pre_time = time.perf_counter()
  while pq:
    priority, current = heapq.heappop(pq)

    if current in visited:
     continue
    visited.add(current)

    if current == goal:
      break
    r, c = current
    neighbours = [(r-1,c),(r+1,c),(r,c-1),(r,c+1)]

    for neighbour in neighbours:
      nr, nc = neighbour

      if not (0<=nr<n_rows and 0 <= nc < n_cols):
        continue
      if grid[nr][nc] == 1:
        continue
      temp_g = g_scores[current] + 1

      if neighbour not in g_scores or temp_g < g_scores[neighbour]:
        g_scores[neighbour] = temp_g
        came_from[neighbour] = current
        new_priority = temp_g + _manhattan(neighbour,goal)
        heapq.heappush(pq,(new_priority, neighbour))
  run_time =time.perf_counter()-pre_time
  nodes_expanded = len(visited)
  if goal not in came_from and goal != start:
    return [], nodes_expanded, run_time
  path = [goal]
  while path[-1] != start:
      path.append(came_from[path[-1]])
  path.reverse()
  return path, nodes_expanded, run_time

In [ ]:
def print_grid(grid):
  for row in grid:
    print(' '.join(str(cell) for cell in row))

def print_grid_with_path(grid, path):
    grid_copy = grid.copy()
    for (r, c) in path:
        grid_copy[r][c] = 2  # or any value that isn't 0 or 1
    print_grid(grid_copy)

Measures Of Evaluation  

In [ ]:
def path_length(path):

  if not path:
    return None
  return len(path)-1

def optimality_ratio(path, start,goal):

  p_length = path_length(path)
  if p_length is None:
    return None
  man = _manhattan(start, goal)
  return p_length/man

In [ ]:
grid = battle_field(70, 'low', start=(0, 0), goal=(69, 69), seed=7)
path = astar(grid, start=(0, 0), goal=(69, 69))
print("Path found:", len(path) > 0)
print("Path length:", len(path) - 1)

Path found: True
Path length: 138


In [ ]:
test_grid = [
        [0, 0, 0, 1, 0],
        [0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 1, 0],
        [0, 0, 0, 1, 0],
    ]
test_path = [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]
print("Path length:", path_length(test_path))
print("Optimality ratio:", optimality_ratio(test_path, (0, 0), (4, 4)))

Path length: 8
Optimality ratio: 1.0


In [ ]:
print_grid(test_grid)

2 2 2 1 0
0 0 2 1 0
0 0 2 2 2
0 0 0 1 2
0 0 0 1 2


In [ ]:
print_grid_with_path(test_grid,test_path)

2 2 2 1 0
0 0 2 1 0
0 0 2 2 2
0 0 0 1 2
0 0 0 1 2



Q3. In the above problem, we relax the condition that all obstacles are Static. In a real world, obstacles can be dynamic and not known a priori. How do you make the UGV navigate and find the optinal path in a dynamic obstacles environment.

Answer: For dynamically changing requirements we will need use dynamic algorithms like d* lite

In [16]:
INF = float('inf')

def get_neighbors(cell, n_rows, n_cols):
    r, c = cell
    candidates = [(r-1,c), (r+1,c), (r,c-1), (r,c+1)]
    return [(nr,nc) for nr, nc in candidates if 0 <= nr < n_rows and 0 <=nc<n_cols]


def cost(grid, a, b):
    br, bc = b
    if grid[br][bc] == 1:
      return INF
    return 1

In [9]:
def calculate_key(s, start, g, rhs,km):
  m = min(g.get(s,INF), rhs.get(s,INF))
  return (m+_manhattan(start,s)+km,m)

In [10]:
def update_vertex(u, goal, grid, g, rhs, U, start, km, n_rows, n_cols):
    if u != goal:
        best = INF
        for s_prime in get_neighbors(u, n_rows, n_cols):
            c = cost(grid, u, s_prime)
            if c < INF:
                val = c + g.get(s_prime, INF)
                if val < best:
                    best = val
        rhs[u] = best
    for i, (k, cell) in enumerate(U):
        if cell == u:
            U[i] = U[-1]
            U.pop()
            heapq.heapify(U)
            break

    if g.get(u, INF) != rhs.get(u, INF):
        heapq.heappush(U, (calculate_key(u, start, g, rhs, km), u))

In [11]:
def compute_shortest_path(goal, start, grid, g, rhs, U, km, n_rows, n_cols):
    while U and (
        U[0][0] < calculate_key(start, start, g, rhs, km)
        or rhs.get(start, INF) != g.get(start, INF)
    ):
        k_old, u = heapq.heappop(U)
        k_new = calculate_key(u, start, g, rhs, km)

        if k_old < k_new:
            heapq.heappush(U, (k_new, u))
        elif g.get(u, INF) > rhs.get(u, INF):
            g[u] = rhs[u]
            for s in get_neighbors(u, n_rows, n_cols):
                update_vertex(s, goal, grid, g, rhs, U, start, km, n_rows, n_cols)
        else:
            g[u] = INF
            update_vertex(u, goal, grid, g, rhs, U, start, km, n_rows, n_cols)
            for s in get_neighbors(u, n_rows, n_cols):
                update_vertex(s, goal, grid, g, rhs, U, start, km, n_rows, n_cols)

In [12]:
def get_path(start, goal, grid, g, n_rows, n_cols, max_steps=10000):
    path = [start]
    current = start
    steps = 0
    while current != goal and steps < max_steps:
        best_next = None
        best_val = INF
        for s in get_neighbors(current, n_rows, n_cols):
            c = cost(grid, current, s)
            if c < INF:
                val = c + g.get(s, INF)
                if val < best_val:
                    best_val = val
                    best_next = s
        if best_next is None or best_val == INF:
            return []  # stuck — no known path with current information
        current = best_next
        path.append(current)
        steps += 1
    return path

In [13]:
def dstar(grid, start, goal):
    g = {}
    rhs = {goal: 0}
    U = []
    km = 0
    heapq.heappush(U, (calculate_key(goal, start, g, rhs, km), goal))
    return g, rhs, U, km


def update_obstacle(cell, is_obstacle, grid, goal, start, last_start, g, rhs, U, km, n_rows, n_cols):
    """Call this whenever the UGV senses a new obstacle (or that a known
    one is gone). Returns the updated (km, last_start) — capture them back
    in your driving loop, since plain values don't mutate in place."""
    grid[cell[0]][cell[1]] = 1 if is_obstacle else 0
    km += _manhattan(last_start, start)
    last_start = start
    for s in get_neighbors(cell, n_rows, n_cols):
        update_vertex(s, goal, grid, g, rhs, U, start, km, n_rows, n_cols)
    update_vertex(cell, goal, grid, g, rhs, U, start, km, n_rows, n_cols)
    compute_shortest_path(goal, start, grid, g, rhs, U, km, n_rows, n_cols)
    return km, last_start

In [17]:
grid = [
    [0, 0, 0, 1, 0],
    [0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 1, 0],
    [0, 0, 0, 1, 0],
]
start, goal = (0, 0), (4, 4)
n_rows, n_cols = len(grid), len(grid[0])

g, rhs, U, km = dstar(grid, start, goal)
compute_shortest_path(goal, start, grid, g, rhs, U, km, n_rows, n_cols)
path = get_path(start, goal, grid, g, n_rows, n_cols)

[(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)] 8


In [18]:
print(path, len(path) - 1)

[(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)] 8


In [19]:
import random
import time

def move_dynamic_obstacles(obstacles, start, goal, n_rows, n_cols):
    new_positions = set()
    for obstacle in obstacles:
        possible_moves = [obstacle]
        for neighbor in get_neighbors(obstacle, n_rows, n_cols):
            if neighbor != start and neighbor != goal:
                possible_moves.append(neighbor)
        new_position = random.choice(possible_moves)
        if new_position != start and new_position != goal and new_position not in new_positions:
            new_positions.add(new_position)
        else:
            new_positions.add(obstacle)
    return new_positions


def detect_obstacles(position, obstacles, sensor_range):
    row, col = position
    detected = set()
    for obstacle in obstacles:
        orow, ocol = obstacle
        if abs(row - orow) + abs(col - ocol) <= sensor_range:
            detected.add(obstacle)
    return detected

In [23]:
def run_dynamic_sim(size=70, start=(0, 0), goal=(69, 69), n_dynamic_obstacles=35,
                     sensor_range=3, max_steps=2000, seed=42):
    random.seed(seed)
    n_rows, n_cols = size, size

    grid = [[0] * size for _ in range(size)]  # UGV starts knowing of NO obstacles
    all_cells = [(r, c) for r in range(size) for c in range(size) if (r, c) != start and (r, c) != goal]
    dynamic_obstacles = set(random.sample(all_cells, n_dynamic_obstacles))

    g, rhs, U, km = dstar(grid, start, goal)
    compute_shortest_path(goal, start, grid, g, rhs, U, km, n_rows, n_cols)

    current = start
    last_start = start
    complete_path = [current]
    replans = 0
    detected_total = set()
    steps = 0
    t0 = time.perf_counter()

    while current != goal and steps < max_steps:
        dynamic_obstacles = move_dynamic_obstacles(dynamic_obstacles, current, goal, n_rows, n_cols)

        newly_detected = detect_obstacles(current, dynamic_obstacles, sensor_range)
        newly_detected = {c for c in newly_detected if grid[c[0]][c[1]] == 0}  # only truly new ones
        detected_total.update(newly_detected)

        for cell in newly_detected:
            km, last_start = update_obstacle(cell, True, grid, goal, current, last_start,
                                              g, rhs, U, km, n_rows, n_cols)
            replans += 1

        path = get_path(current, goal, grid, g, n_rows, n_cols)
        if not path or len(path) < 2:
            steps += 1
            continue

        next_position = path[1]
        if next_position in dynamic_obstacles:  # obstacle moved into our next step
            steps += 1
            continue

        current = next_position
        complete_path.append(current)
        steps += 1

    runtime = time.perf_counter() - t0

    return {
        "Path found": current == goal,
        "Path length": len(complete_path) - 1,
        "Steps": steps,
        "Replanning operations": replans,
        "Obstacles detected": len(detected_total),
        "Runtime (sec)": round(runtime, 4),
        "Path": complete_path,
    }

In [24]:
result = run_dynamic_sim()
for k, v in result.items():
    if k != "Path":
        print(f"{k}: {v}")

Path found: True
Path length: 138
Steps: 138
Replanning operations: 14
Obstacles detected: 14
Runtime (sec): 0.0301
